In [ ]:
import pandas as pd
import os
from datetime import timedelta

In [ ]:
basins = {
    'Deraniyagala': 'Wet',
    'Nawalapitiya': 'Wet',
    'Padiyathalawa': 'Dry',
    'Wellawaya': 'Dry'
}

paths = {
    'Deraniyagala': r"D:\Research Datasets\Additional Datasets\Working Folder\Four basins\ICESWSE_Results\Deraniyagala_SPI_SSI.xlsx",
    'Nawalapitiya': r"D:\Research Datasets\Additional Datasets\Working Folder\Four basins\ICESWSE_Results\Nawalapitiya_SPI_SSI.xlsx",
    'Padiyathalawa': r"D:\Research Datasets\Additional Datasets\Working Folder\Four basins\ICESWSE_Results\Padiyathalawa_SPI_SSI.xlsx",
    'Wellawaya': r"D:\Research Datasets\Additional Datasets\Working Folder\Four basins\ICESWSE_Results\Wellawaya_SPI_SSI.xlsx"
}

In [ ]:
MAX_LAG_BY_ZONE = {
    "Wet": {1: 3, 3: 6, 12: 9},
    "Dry": {1: 4, 3: 8, 12: 12}
}

timescales = [1, 3, 12]
thresholds = [-1.0, -1.5, -2.0]

In [ ]:
base_dir = r"D:\Research Datasets\Additional Datasets\Working Folder\Four basins\ICESWSE_Results"
output_dir = os.path.join(base_dir, "Detailed_Drought_Analysis")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "Detailed_Propagation_Results_fourbasins.xlsx")

In [ ]:
spi_all = []
ssi_all = []
paired_all = []
rates_all = []

In [ ]:
for basin, zone in basins.items():
    df = pd.read_excel(paths[basin], parse_dates=['Date'])
    
    for ts in timescales:
        spi_col = f'SPI_{ts}'
        ssi_col = f'SSI_{ts}'
        max_lag = MAX_LAG_BY_ZONE[zone][ts]

In [ ]:
 for th in thresholds:
            # SPI events
            df['is_drought_spi'] = df[spi_col] < th
            df['group_spi'] = (df['is_drought_spi'] != df['is_drought_spi'].shift()).cumsum()
            spi_events = df[df['is_drought_spi']].groupby('group_spi').agg(
                start=('Date', 'first'),
                end=('Date', 'last'),
                duration=('Date', 'size'),
                severity=(spi_col, lambda x: -x.sum()),
                intensity=(spi_col, lambda x: -x.mean()),
                peak=(spi_col, 'min'),
                peak_date=(spi_col, lambda x: df.loc[x.idxmin(), 'Date'])
            ).reset_index(drop=True)
            spi_events[['basin', 'timescale', 'threshold']] = [basin, ts, th]
            spi_all.append(spi_events)

In [ ]:
            df['is_drought_ssi'] = df[ssi_col] < th
            df['group_ssi'] = (df['is_drought_ssi'] != df['is_drought_ssi'].shift()).cumsum()
            ssi_events = df[df['is_drought_ssi']].groupby('group_ssi').agg(
                start=('Date', 'first'),
                end=('Date', 'last'),
                duration=('Date', 'size'),
                severity=(ssi_col, lambda x: -x.sum()),
                intensity=(ssi_col, lambda x: -x.mean()),
                peak=(ssi_col, 'min'),
                peak_date=(ssi_col, lambda x: df.loc[x.idxmin(), 'Date'])
            ).reset_index(drop=True)
            ssi_events[['basin', 'timescale', 'threshold']] = [basin, ts, th]
            ssi_all.append(ssi_events)

In [ ]:
 paired = []
            used_hd = set()
            
            for _, md in spi_events.iterrows():
                max_end = md['start'] + timedelta(days=30 * max_lag)
                candidates = ssi_events[
                    (ssi_events['start'] >= md['start']) &
                    (ssi_events['start'] <= max_end) &
                    (ssi_events['peak_date'] >= md['peak_date']) &
                    (ssi_events['end'] >= md['end']) &
                    (~ssi_events.index.isin(used_hd))
                ]

In [ ]:
 if not candidates.empty:
                    hd = candidates.sort_values('start').iloc[0]
                    used_hd.add(hd.name)
                    onset_lag = (hd['start'] - md['start']).days / 30.0

                    peak_lag = (hd['peak_date'] - md['peak_date']).days / 30.0

                    recovery_lag = (hd['end'] - md['end']).days / 30.0

In [ ]:
                    paired_row = {
                        'basin': basin,
                        'timescale': ts,
                        'threshold': th,
                        'md_start': md['start'],
                        'md_end': md['end'],
                        'md_duration': md['duration'],
                        'md_severity': md['severity'],
                        'md_intensity': md['intensity'],
                        'md_peak': md['peak'],
                        'md_peak_date': md['peak_date'],
                        'hd_start': hd['start'],
                        'hd_end': hd['end'],
                        'hd_duration': hd['duration'],
                        'hd_severity': hd['severity'],
                        'hd_intensity': hd['intensity'],
                        'hd_peak': hd['peak'],
                        'hd_peak_date': hd['peak_date'],
                        'onset_lag': onset_lag,
                        'peak_lag': peak_lag,
                        'recovery_lag': recovery_lag
                    }
                    paired.append(paired_row)

In [ ]:
 if paired:
                paired_df_temp = pd.DataFrame(paired)
                paired_all.append(paired_df_temp)

In [ ]:
total_spi = len(spi_events)
            total_ssi = len(ssi_events)
            matched = len(paired)
            rate = (matched / total_spi * 100) if total_spi else 0
            
            rates_all.append({
                'basin': basin,
                'timescale': ts,
                'threshold': th,
                'total_spi_events': total_spi,
                'total_ssi_events': total_ssi,
                'matched_pairs': matched,
                'num_onset_lags': matched,
                'num_peak_lags': matched,
                'num_recovery_lags': matched,
                'matching_rate_%': rate
            })

In [ ]:
spi_df = pd.concat([df for df in spi_all if not df.empty], ignore_index=True) if spi_all else pd.DataFrame()
ssi_df = pd.concat([df for df in ssi_all if not df.empty], ignore_index=True) if ssi_all else pd.DataFrame()
paired_df = pd.concat(paired_all, ignore_index=True) if paired_all else pd.DataFrame()
rates_df = pd.DataFrame(rates_all)

In [ ]:
if not paired_df.empty:
    prop_summary = paired_df.groupby(['basin', 'timescale', 'threshold']).agg({
        'onset_lag': ['count', 'mean', 'min', 'max', 'median'],
        'peak_lag': ['count', 'mean', 'min', 'max', 'median'],
        'recovery_lag': ['count', 'mean', 'min', 'max', 'median'],
        'md_duration': 'mean',
        'hd_duration': 'mean',
        'md_severity': 'mean',
        'hd_severity': 'mean',
        'md_intensity': 'mean',
        'hd_intensity': 'mean'
    }).reset_index()

In [ ]:
  prop_summary.columns = ['_'.join(col).strip() if col[1] else col[0] for col in prop_summary.columns.values]
    prop_summary.rename(columns={
        'onset_lag_count': 'num_pairs_onset',
        'peak_lag_count': 'num_pairs_peak',
        'recovery_lag_count': 'num_pairs_recovery'
    }, inplace=True)

In [ ]:
    lag_stats = paired_df.groupby(['basin', 'timescale', 'threshold']).agg({
        'onset_lag': ['count', 'mean', 'min', 'max', 'median'],
        'peak_lag': ['count', 'mean', 'min', 'max', 'median'],
        'recovery_lag': ['count', 'mean', 'min', 'max', 'median']
    }).reset_index()
    
    lag_stats.columns = ['_'.join(col).strip() if col[1] else col[0] for col in lag_stats.columns.values]
else:
    prop_summary = pd.DataFrame()
    lag_stats = pd.DataFrame()

In [ ]:
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    spi_df.to_excel(writer, sheet_name='SPI_Events', index=False)
    ssi_df.to_excel(writer, sheet_name='SSI_Events', index=False)
    paired_df.to_excel(writer, sheet_name='Paired_Events', index=False)
    rates_df.to_excel(writer, sheet_name='Event_Matching_Summary', index=False)
    prop_summary.to_excel(writer, sheet_name='Propagation_Summary', index=False)
    lag_stats.to_excel(writer, sheet_name='Lag_Statistics', index=False)